# 05: AM defense (PLAN.md §6, §13.7), trimmed to ~4 quota-hours
Build AM on BASE, choose its recipe **on dev only**, freeze it. One resumable runner does it all
(`aftermarket.defense.pipeline`); the first thing it prints is the estimated cost per stage.

1. **locate** L* for k = 4, 6, 8 (refusal-direction strength, SafeTune safety neurons, CircuitKIT EAP-IG with a CPU
   fallback; mean-rank merge) and the method-overlap table for the write-up.
2. **harden** two candidates at k = 6, spread-refusal off / on: **SafeGrad (plain-loss variant, T4 memory)**, LoRA
   on L* only, 300 refusal + 300 benign pairs, compression shield (L* at 8-bit under A4) on both.
3. **dev evaluation**: A0, A3, A4q3 on dev splits, judged with WildGuard. A2 is *not* in the selection loop
   (a fine-tune per candidate costs too much quota); the frozen AM still faces A2 on test.
4. **select**: best mean retention (A3, A4q3) with benign_dev over-refusal <= BASE + 3 pts and mmlu_dev >= BASE - 2 pts.
5. **controls** with the chosen recipe, same budget: AM_rand0, AM_rand1 (random layers outside L*), AM_all.
6. **freeze**: AM + controls pushed to the PRIVATE store `<you>/aftermarket-am`, revision pinned in
   `configs/am_frozen.yaml`.

ReSta re-anchoring was dropped (see `notes/limitations.md`). The AM test battery (A0-A5 on test, `--final`) is
budgeted separately: set `RUN_AM_BATTERY = True` to append it to this run.

**Save & Run All (Commit)** safe: every stage skips finished work and pulls from the private Hub first, and the
runner stops cleanly after the stage that crosses `MAX_HOURS`, so just commit again to continue.
If no candidate is eligible, nothing is frozen and nothing runs on test (report it honestly).

Requires: `HF_TOKEN` (write), GPU T4 x2.

In [ ]:
# Session clock for the --max-hours guard (Kaggle charges the whole session, setup included)
import time
SESSION_START = time.time()
MAX_HOURS = 4.0          # stop cleanly after the stage that crosses this
RUN_AM_BATTERY = False   # True: also run the frozen AM family through A0-A5 on test (--final), budgeted separately

In [ ]:
# (a) Clone the repo (or update it when re-running in the same session). Non-interactive.
import os, subprocess
REPO_URL = "https://github.com/Bennytimz/aftermarket"
if os.path.basename(os.getcwd()) != "aftermarket":
    if not os.path.isdir("aftermarket/.git"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("aftermarket")
subprocess.run(["git", "pull", "-q", "--ff-only"], check=False)
print("repo at", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
# (b) Install only what Kaggle lacks (requirements-kaggle.txt), keeping Kaggle's torch build fixed.
#     torchao is removed: Kaggle's 0.10.0 breaks PEFT imports and we don't use it.
!pip uninstall -y -q torchao
!pip freeze | grep -iE '^(torch|torchvision|torchaudio)==' > /tmp/keep-torch.txt
!pip install -q -r requirements-kaggle.txt -c /tmp/keep-torch.txt
import importlib.metadata as md
for p in ["torch", "transformers", "peft", "accelerate", "huggingface_hub", "datasets", "bitsandbytes",
          "hqq", "curatorkit", "safetune", "circuitkit", "auditkit"]:
    try: print(f"{p:16s}", md.version(p))
    except md.PackageNotFoundError: print(f"{p:16s} MISSING")

In [ ]:
# (c) Log in to Hugging Face from Kaggle secrets (works in background "Save & Run All" runs; never prompts).
# Attach the secret HF_TOKEN (Add-ons -> Secrets) before committing.
from huggingface_hub import login, whoami
HF_USER = None
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN"), add_to_git_credential=False)
    HF_USER = whoami()["name"]
    print("HF user:", HF_USER)
except Exception as e:
    print(f"WARNING: no HF login ({type(e).__name__}: {e}). Gated models will fail; pushes/pulls are skipped.")

In [ ]:
# (d) GPU info
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB")

In [ ]:
# (e) Pinned model revision
BASE_MODEL = "CohereLabs/tiny-aya-global"
BASE_REVISION = "af89d219b53ed9b13b8a4645f9c8028973510324"

In [ ]:
import os, json, sys
os.environ["AFTERMARKET_DEFENSE_DIR"] = "/kaggle/temp/aftermarket_defense" if os.path.isdir("/kaggle") else "/tmp/aftermarket_defense"
os.environ["AFTERMARKET_ADAPTERS"] = "/kaggle/temp/aftermarket_adapters" if os.path.isdir("/kaggle") else "/tmp/aftermarket_adapters"
os.environ["AFTERMARKET_TMP"] = "/kaggle/temp" if os.path.isdir("/kaggle") else "/tmp"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
for k in ("AFTERMARKET_DEFENSE_DIR", "AFTERMARKET_ADAPTERS"):
    os.makedirs(os.environ[k], exist_ok=True)
sys.path.insert(0, "src")
DDIR = os.environ["AFTERMARKET_DEFENSE_DIR"]
BATTERY_FLAG = "--with-battery" if RUN_AM_BATTERY else ""
def read_json(path):
    try:
        return json.load(open(path))
    except Exception:
        return None
print("defense dir:", DDIR, "| max hours:", MAX_HOURS, "| battery:", RUN_AM_BATTERY)

## Estimated cost per stage (measured timings from earlier runs where available, else a-priori)

In [ ]:
!PYTHONPATH=src python -m aftermarket.defense.pipeline --config configs/defense.yaml --plan-only --max-hours {MAX_HOURS} --session-start {SESSION_START} {BATTERY_FLAG}

## Run (resumable; stops cleanly after the stage that crosses MAX_HOURS)

In [ ]:
!PYTHONPATH=src python -m aftermarket.defense.pipeline --config configs/defense.yaml --max-hours {MAX_HOURS} --session-start {SESSION_START} {BATTERY_FLAG}

In [ ]:
# Final summary (always runs; stats only, no completion text)
try:
    print(f"session time used: {(time.time() - SESSION_START) / 3600:.2f} h of {MAX_HOURS}")
    loc = read_json(f"{DDIR}/locate/locate.json")
    if loc:
        print("locate methods:", loc["methods_used"], "| failures:", loc["failures"] or "none")
        for k, L in loc["L_star"].items():
            print(f"  L*_{k} = {L} | overlap {loc['overlap'][k]['jaccard']} | in all: {loc['overlap'][k]['in_all']}")
    else:
        print("locate: not done")
    from aftermarket.defense.store import local_specs
    for mid, s in local_specs().items():
        m = s["defense"]
        print(f"  {mid:10s} {m.get('kind')} layers={m.get('layers') if len(m.get('layers') or []) <= 8 else 'all'} "
              f"spread={m.get('spread')} params={m.get('trainable_params')} train_s={m.get('train_seconds')} "
              f"loss {m.get('loss_first')} -> {m.get('loss_last')} shield={bool(m.get('quant_shield'))} | {m.get('method')}")
    sel = read_json(f"{DDIR}/selection.json")
    if sel:
        for t in sel["stage1"]["table"]:
            print(f"  {t['id']:10s} retention {t['mean_retention']} {t['retention']} | A0 ASR {t['asr_A0']} | "
                  f"over-refusal {t['over_refusal']} | mmlu {t['mmlu']} | {'eligible' if t['eligible'] else 'x ' + ','.join(t['why_not'])}")
        print("recipe:", sel.get("recipe"))
    if os.path.exists("configs/am_frozen.yaml"):
        import yaml
        fz = yaml.safe_load(open("configs/am_frozen.yaml"))
        print("FROZEN:", fz["recipe"], "| L* =", fz["L_star"], "|", fz["store"])
        print("commit configs/am_frozen.yaml to git (content below):")
        print(open("configs/am_frozen.yaml").read())
    else:
        print("AM not frozen yet")
except Exception as e:
    print(f"final summary failed: {type(e).__name__}: {e}")